## Setup

In [ ]:
import json
from pathlib import Path
from string import ascii_uppercase

import matplotlib.pyplot as plt
import numpy as np
import scipy.io as sio
import torch
from tigramite import data_processing as pp
from tigramite import plotting as tp
from tigramite.independence_tests.gsquared import Gsquared
from tigramite.lpcmci import LPCMCI

from csi_vae_gumbel.dataset import CSIDataset
from csi_vae_gumbel.models.vae import Parameters
from csi_vae_gumbel.models.vae.single_antenna_vae import SingleAntennaVAE
from csi_vae_gumbel.settings import Settings

settings = Settings()

ACTIVITIES_IDS = [f"S1a_{x}" for x in ascii_uppercase[: settings.n_activities]]
ACTIVITIES_LABELS = settings.activities
TIME_STEP = 15
GPU_ID = 0

/mnt/servicesdata/lcotti/csi-vae-gumbel/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset

We use the complete dataset for causal inference, withouy any train/test split.

In [ ]:
files = [Path(f"../{settings.dataset_path}") / f"S1a_{x}.mat" for x in ascii_uppercase[: settings.n_activities]]
mats = [np.array(sio.loadmat(file)["csi"]) for file in files]

dataset = CSIDataset(
    mats,
    settings.test_window_size,
    n_antennas=1,
    antenna_select=0,
    augment_probability=0,
)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=len(dataset), shuffle=False)

In [ ]:
def get_best_model() -> tuple[SingleAntennaVAE, Parameters]:
    """Load the best model and its parameters from the study results."""
    with Path(f"{settings.study_path}/study_results.json").open("r") as f:
        study_info = json.load(f)

    best_model_path = Path(settings.study_path) / f"trial_{study_info['best_trial']}"

    with Path(best_model_path / "results.json").open("r") as f:
        info = json.load(f)

    params = Parameters(
        final_cap=info["final_cap"],
        start_gumbel_temp=info["start_gumbel_temp"],
        final_kl_weight=info["final_kl_weight"],
        latent_dim=info["latent_dim"],
    )

    vae_model = SingleAntennaVAE(
        settings.train_window_size,
        settings.n_subcarriers,
        settings.n_categories,
        params.latent_dim,
    )
    best_model_weights = torch.load(best_model_path / "model.pt")
    vae_model.load_state_dict(best_model_weights)

    return vae_model, params

vae, params = get_best_model()
vae.eval()
vae = vae.to(GPU_ID)

In [ ]:
all_latents = []
all_labels = []

# 1. Convert to Categorical Indices first (T, latent_dim)
with torch.no_grad():
    for x, y in dataloader:
        _, z_hard, _ = vae(x.to(GPU_ID))
        # Convert one-hot to index: (B, latent_dim, n_categories) -> (B, latent_dim)
        z_cat = z_hard.argmax(dim=-1)

        all_latents.append(z_cat.cpu().numpy())
        all_labels.append(y.cpu().numpy())

latents = np.concatenate(all_latents, axis=0)
labels = np.concatenate(all_labels, axis=0)

labelled_causal_data = {}

# 2. Split and THEN shift per activity
for label in np.unique(labels):
    # Extract latents for this specific movement sequence
    activity_latents = latents[labels == label]

    # Create the shifted views locally for this activity
    # This ensures t-1 and t-2 belong to the SAME activity
    data_t = activity_latents[2:]  # current
    data_t_1 = activity_latents[1:-1]  # lag 1
    data_t_2 = activity_latents[:-2]  # lag 2

    # Horizontal stack: (T-2, latent_dim * 3)
    # Row format: [Z_0...Z_dim (t), Z_0...Z_dim (t-1), Z_0...Z_dim (t-2)]
    activity_causal_data = np.hstack([data_t, data_t_1, data_t_2])

    labelled_causal_data[int(label)] = activity_causal_data

## Causal Analysis

In [ ]:
# 1. Setup the analysis parameters
# Use Gsquared for categorical (discrete) data
g_test = Gsquared(significance="exact")
tau_max = 2

# Correct names: We have 'latent_dim' variables, each taking 'n_categories' values
var_names = [f"Z{i}" for i in range(params.latent_dim)]

# 2. Iterate through each activity
for label_id, data in labelled_causal_data.items():
    # 'data' should be shape (T, latent_dim) containing the argmax indices
    activity_name = ACTIVITIES_LABELS[label_id]

    if len(data) <= tau_max:
        print(f"Skipping {activity_name}: Not enough samples.")
        continue

    # Initialize Tigramite DataFrame
    # Note: 'datatypes' should be 'discrete' for Gsquared
    dataframe = pp.DataFrame(data.astype(np.int32), var_names=var_names, data_type=np.ones_like(data, dtype=int))

    # 3. Run LPCMCI
    # LPCMCI is robust to latent common causes (unobserved drivers in CSI)
    lpcmci = LPCMCI(dataframe=dataframe, cond_ind_test=g_test, verbosity=0)

    # pc_alpha is the significance level for the conditional independence tests
    results = lpcmci.run_lpcmci(tau_max=tau_max, pc_alpha=0.05)

    # 4. Plotting
    # Time series graph is best for t, t-1, t-2 visualization
    fig, ax = tp.plot_time_series_graph(
        val_matrix=results["val_matrix"],
        graph=results["graph"],
        var_names=var_names,
        link_colorbar_label="MCI Strength",
    )

    plt.title(f"Temporal Latent Causal Graph: {activity_name}")
    plt.show()